### FEMA -- Data Preprocessing

- Fix data quality issues and prepare clean datasets for feature engineering & modelling

### Setup & import libraries


In [5]:

import os, warnings
import pandas as pd
import numpy as np

warnings.filterwarnings("ignore")

## Define raw and processed paths

DATA_RAW = os.path.join("..", "data", "raw")
DATA_PROCESSED = os.path.join("..", "data", "processed")

os.makedirs(DATA_PROCESSED, exist_ok=True)

## Load the datasets

declarations = pd.read_csv(os.path.join(DATA_RAW, "declarations.csv"))
public_assistance = pd.read_csv(os.path.join(DATA_RAW, "public_assistance.csv"))
disaster_summaries = pd.read_csv(os.path.join(DATA_RAW, "disaster_summaries.csv"))

In [6]:

# Standardize column names

for name, df in [
    ("declarations", declarations),
    ("public_assistance", public_assistance),
    ("disaster_summaries", disaster_summaries)
]:
    df.columns = (
        df.columns
        .str.strip()
        .str.replace(" ", "_")
        .str.replace("-", "_")
        .str.lower()
    )
    print(f"{name} columns cleaned")



declarations columns cleaned
public_assistance columns cleaned
disaster_summaries columns cleaned


### Step 1 - Cleaning Exercise: Declarations Dataset

In [7]:
print(declarations.dtypes)
print(declarations.isnull().sum())
declarations.head()

disasternumber               int64
state                       object
declarationtype             object
incidenttype                object
declarationdate             object
incidentbegindate           object
incidentenddate             object
fydeclared                   int64
designatedarea              object
declarationrequestnumber     int64
dtype: object
disasternumber                0
state                         0
declarationtype               0
incidenttype                  0
declarationdate               0
incidentbegindate             0
incidentenddate             534
fydeclared                    0
designatedarea                0
declarationrequestnumber      0
dtype: int64


,disasternumber,state,declarationtype,incidenttype,declarationdate,incidentbegindate,incidentenddate,fydeclared,designatedarea,declarationrequestnumber
0,3610,PR,EM,Severe Storm,2024-08-13T00:00:00.000Z,2024-08-13T00:00:00.000Z,2024-08-16T00:00:00.000Z,2024,Adjuntas (Municipio),24124
1,5529,OR,FM,Fire,2024-08-09T00:00:00.000Z,2024-08-08T00:00:00.000Z,NaN,2024,Washington (County),24122
2,5528,OR,FM,Fire,2024-08-06T00:00:00.000Z,2024-08-04T00:00:00.000Z,NaN,2024,Jefferson (County),24116
3,5527,OR,FM,Fire,2024-08-02T00:00:00.000Z,2024-08-02T00:00:00.000Z,NaN,2024,Deschutes (County),24111
4,3610,PR,EM,Severe Storm,2024-08-13T00:00:00.000Z,2024-08-13T00:00:00.000Z,2024-08-16T00:00:00.000Z,2024,Aguada (Municipio),24124


In [8]:
# save a copy of dataframe
declarations_clean = declarations.copy()


# STEP 1a - fix the date columns (convert to datetime)("declarationdate","incidentbegindate","incidentenddate")

for col in ["declarationdate","incidentbegindate","incidentenddate"]:
    if col in declarations_clean.columns:
        declarations_clean[col] = (
            pd.to_datetime(declarations_clean[col], utc=True,
            errors="coerce"
        ).dt.tz_localize(None)
    )
    

# print all attributes
print(declarations_clean.dtypes)
print(declarations_clean.isnull().sum())
declarations_clean.head()
    

disasternumber                       int64
state                               object
declarationtype                     object
incidenttype                        object
declarationdate             datetime64[ns]
incidentbegindate           datetime64[ns]
incidentenddate             datetime64[ns]
fydeclared                           int64
designatedarea                      object
declarationrequestnumber             int64
dtype: object
disasternumber                0
state                         0
declarationtype               0
incidenttype                  0
declarationdate               0
incidentbegindate             0
incidentenddate             534
fydeclared                    0
designatedarea                0
declarationrequestnumber      0
dtype: int64


,disasternumber,state,declarationtype,incidenttype,declarationdate,incidentbegindate,incidentenddate,fydeclared,designatedarea,declarationrequestnumber
0,3610,PR,EM,Severe Storm,2024-08-13,2024-08-13,2024-08-16,2024,Adjuntas (Municipio),24124
1,5529,OR,FM,Fire,2024-08-09,2024-08-08,NaT,2024,Washington (County),24122
2,5528,OR,FM,Fire,2024-08-06,2024-08-04,NaT,2024,Jefferson (County),24116
3,5527,OR,FM,Fire,2024-08-02,2024-08-02,NaT,2024,Deschutes (County),24111
4,3610,PR,EM,Severe Storm,2024-08-13,2024-08-13,2024-08-16,2024,Aguada (Municipio),24124


### Step 2 - Cleaning Exercise: Public Assistance Dataset

In [9]:
print(f"Public Assistance columns data-types:\n{public_assistance.dtypes}") 
print(public_assistance.isnull().sum())
print(public_assistance.describe())

Public Assistance columns data-types:
disasternumber             int64
stateabbreviation         object
projectamount            float64
federalshareobligated    float64
totalobligated           float64
projectsize               object
damagecategorycode        object
applicationtitle          object
dtype: object
disasternumber           0
stateabbreviation        0
projectamount            0
federalshareobligated    0
totalobligated           0
projectsize              0
damagecategorycode       0
applicationtitle         0
dtype: int64
       disasternumber  projectamount  federalshareobligated  totalobligated
count   812867.000000   8.128670e+05           8.128670e+05    8.128670e+05
mean      2776.388442   3.744829e+05           3.482599e+05    3.488927e+05
std       1325.805880   1.384843e+07           1.347969e+07    1.348428e+07
min       1239.000000  -3.726871e+08          -3.726871e+08   -3.764233e+08
25%       1609.000000   4.410000e+03           3.502495e+03    3.561990e+03

In [10]:
# make a copy of original datframe
pa_clean = public_assistance.copy()

# STEP 2a - Clean funding columns

funding_cols = [
    "projectamount",
    "federalshareobligated",
    "totalobligated"
]

for col in funding_cols:
    if col in pa_clean.columns:
        pa_clean[col] = pd.to_numeric(pa_clean[col], errors="coerce")
        pa_clean[col] = pa_clean[col].fillna(0) # fill missing funding with zero 
        pa_clean[col] = pa_clean[col].clip(lower=0) # removes any negative funding 

print(f"pa_clean : {pa_clean.shape}")
pa_clean.head()

pa_clean : (812867, 8)


,disasternumber,stateabbreviation,projectamount,federalshareobligated,totalobligated,projectsize,damagecategorycode,applicationtitle
0,1361,WA,1203.00,902.25,957.10,Small,B,(PW# 81) INSPECTION OF TOWN BUILDINGS & UTILITIES
1,1603,LA,15156787.07,15156787.07,15312129.35,Large,E,(PW# 18773) CONTENTS ROLLUP-MULTIPLE SITES; MU...
2,3582,MS,159127.24,119345.43,119345.43,Small,B,MS- State Department of Health EPM-Period 1 - ...
3,1817,WA,6847.49,5135.62,5135.62,Small,C,(PW# 537) SEBC005 - S. Emery Ave. Road/Shoulde...
4,3582,MS,118703.32,89027.49,89027.49,Small,B,MS- State Department of Health EPM-Period 3 - ...



### Step 3 - Cleaning Exercise: Disaster Summaries Dataset

In [11]:
# overview
print(f"Disaster Summaries shape: {disaster_summaries.shape}")
print(f"Disaster Summaries columns data-types:\n{disaster_summaries.dtypes}")
print("\nMissing values:")
print(disaster_summaries.isnull().sum())
disaster_summaries.head()

Disaster Summaries shape: (3945, 11)
Disaster Summaries columns data-types:
disasternumber                  int64
totalnumberiaapproved         float64
totalamountihpapproved        float64
totalamounthaapproved         float64
totalamountonaapproved        float64
totalobligatedamountpa        float64
totalobligatedamountcatab     float64
totalobligatedamountcatc2g    float64
totalobligatedamounthmgp      float64
paloaddate                     object
ialoaddate                     object
dtype: object

Missing values:
disasternumber                   0
totalnumberiaapproved         3339
totalamountihpapproved        3339
totalamounthaapproved         3395
totalamountonaapproved        3341
totalobligatedamountpa         956
totalobligatedamountcatab     1217
totalobligatedamountcatc2g    2389
totalobligatedamounthmgp      1102
paloaddate                     956
ialoaddate                    3339
dtype: int64


,disasternumber,totalnumberiaapproved,totalamountihpapproved,totalamounthaapproved,totalamountonaapproved,totalobligatedamountpa,totalobligatedamountcatab,totalobligatedamountcatc2g,totalobligatedamounthmgp,paloaddate,ialoaddate
0,3601,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,3602,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1802,NaN,NaN,NaN,NaN,1.881275e+07,1.342309e+07,4.869890e+06,2710679.0,2026-02-06T00:00:00.000Z,NaN
3,1292,NaN,NaN,NaN,NaN,2.981058e+08,1.436027e+08,1.504852e+08,67665966.0,2026-02-06T00:00:00.000Z,NaN
4,3163,NaN,NaN,NaN,NaN,5.380816e+06,5.369404e+06,NaN,NaN,2026-02-06T00:00:00.000Z,NaN


In [12]:
# make a copy of original datafram
summaries_clean = disaster_summaries.copy()

# STEP 3b - fix date columns

for col in ["paloaddate", "ialoaddate"]:
    if col in summaries_clean.columns:
        summaries_clean[col] = (
        pd.to_datetime(summaries_clean[col], utc=True,
            errors="coerce"
        ).dt.tz_localize(None)
    )
    
    summaries_clean = summaries_clean.dropna(subset=["paloaddate", "ialoaddate"]) 



In [13]:

# STEP 3c - Clean funding cost

for col in summaries_clean.select_dtypes(include=["int64", "float64"]).columns:
    if col in summaries_clean.columns:
        summaries_clean[col] = (
            pd.to_numeric(summaries_clean[col], errors="coerce"))
        summaries_clean[col] = summaries_clean[col].fillna(0) # fill missing funding with 0

In [14]:
# STEP 3d  print out all attributes
    
summaries_clean = summaries_clean.drop_duplicates()
print(f"Disaster Summaries columns data-types:\n{summaries_clean.dtypes}")
print("\nMissing values:")
print(summaries_clean.isnull().sum())

summaries_clean.head()


Disaster Summaries columns data-types:
disasternumber                         int64
totalnumberiaapproved                float64
totalamountihpapproved               float64
totalamounthaapproved                float64
totalamountonaapproved               float64
totalobligatedamountpa               float64
totalobligatedamountcatab            float64
totalobligatedamountcatc2g           float64
totalobligatedamounthmgp             float64
paloaddate                    datetime64[ns]
ialoaddate                    datetime64[ns]
dtype: object

Missing values:
disasternumber                0
totalnumberiaapproved         0
totalamountihpapproved        0
totalamounthaapproved         0
totalamountonaapproved        0
totalobligatedamountpa        0
totalobligatedamountcatab     0
totalobligatedamountcatc2g    0
totalobligatedamounthmgp      0
paloaddate                    0
ialoaddate                    0
dtype: int64


,disasternumber,totalnumberiaapproved,totalamountihpapproved,totalamounthaapproved,totalamountonaapproved,totalobligatedamountpa,totalobligatedamountcatab,totalobligatedamountcatc2g,totalobligatedamounthmgp,paloaddate,ialoaddate
3332,1971,16375.0,77097372.06,57302364.20,19795007.86,2.010424e+08,1.271363e+08,6.708809e+07,63346282.14,2026-02-06,2026-05-26
3333,4420,3428.0,27279186.25,24356998.74,2922187.51,5.294109e+08,5.027339e+07,4.440429e+08,38797583.91,2026-05-26,2026-05-26
3334,4531,5577.0,37357961.48,0.00,37357961.48,5.647922e+08,5.605798e+08,0.000000e+00,15288527.33,2026-05-26,2026-05-26
3335,4534,1940.0,10616929.18,0.00,10616929.18,2.256010e+08,2.249514e+08,0.000000e+00,1463754.80,2026-05-26,2026-05-26
3336,1925,1624.0,10602928.54,9705873.50,897055.04,5.930566e+06,1.577961e+06,4.144901e+06,2821027.04,2026-02-06,2026-05-26


### Step 4 - Referential Integrity

In [15]:
# step 4a - validate disaster dataset
valid_disasters = set(declarations_clean["disasternumber"])

# step 4b - check the public assistance and summaries dataset
before_pa = len(pa_clean)
before_summaries = len(summaries_clean)

# step 4c - filter  to check if public assistance and summaries are in valid disaster
pa_clean = pa_clean[pa_clean["disasternumber"].isin(valid_disasters)]
summaries_clean = summaries_clean[summaries_clean["disasternumber"].isin(valid_disasters)]

print(f"Public Assistance rows dropped: {before_pa - len(pa_clean)}")
print(f"Disaster Summary rows dropped: {before_summaries - len(summaries_clean)}")
print("Referential integrity check completed")

Public Assistance rows dropped: 0
Disaster Summary rows dropped: 0
Referential integrity check completed


###  Step 5 - Validate Row Counts

In [16]:
print("Row count - raw vs clean")

for name, raw, clean in [
    ("declarations", declarations, declarations_clean),
    ("public_assistance", public_assistance, pa_clean),
    ("disaster_summaries", disaster_summaries, summaries_clean)
]:
    print(f"{name:<22}: {len(raw):>7,} -> {len(clean):>7,}")

Row count - raw vs clean
declarations          :  69,905 ->  69,905
public_assistance     : 812,867 -> 812,867
disaster_summaries    :   3,945 ->     527


### Step 6 - Save Clean Datasets

In [17]:
files = {
    "declarations_clean.csv": declarations_clean,
    "public_assistance_clean.csv": pa_clean,
    "disaster_summaries_clean.csv": summaries_clean
}

for fname, df in files.items():
    df.to_csv(os.path.join(DATA_PROCESSED, fname), index=False)
    print(f"{fname} saved successfully")

declarations_clean.csv saved successfully
public_assistance_clean.csv saved successfully
disaster_summaries_clean.csv saved successfully
